# 03 — Train Random Forest Classifier

Tune the required Random Forest parameter space using RandomizedSearchCV, then save the final model artifact.

In [ ]:
import pandas as pd, joblib
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
FEATURE_BASE = ['radius','texture','perimeter','area','smoothness','compactness','concavity','concave_points','symmetry','fractal_dimension']
FEATURES = [f'{stat}_{feat}' for stat in ['mean','se','worst'] for feat in FEATURE_BASE]
RANDOM_STATE=42
df=pd.read_csv('data/wdbc.data',header=None,names=['id','diagnosis']+FEATURES)
X=df[FEATURES]; y=(df.diagnosis=='M').astype(int)
X_train,X_temp,y_train,y_temp=train_test_split(X,y,test_size=.30,stratify=y,random_state=RANDOM_STATE)
X_val,X_test,y_val,y_test=train_test_split(X_temp,y_temp,test_size=.50,stratify=y_temp,random_state=RANDOM_STATE)

In [ ]:
param_grid={
 'n_estimators':[200,500,800],
 'max_depth':[None,5,10,20],
 'min_samples_split':[2,5,10],
 'min_samples_leaf':[1,2,4],
 'max_features':['sqrt',0.5,1.0],
 'class_weight':[None,'balanced']
}
rf=RandomForestClassifier(random_state=RANDOM_STATE,n_jobs=-1)
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=RANDOM_STATE)
search=RandomizedSearchCV(rf,param_grid,n_iter=6,scoring='roc_auc',cv=cv,random_state=RANDOM_STATE,n_jobs=-1,refit=True,return_train_score=True)
search.fit(X_train,y_train)
print('Best params:',search.best_params_)
print('Best CV ROC-AUC:',search.best_score_)

In [ ]:
model=search.best_estimator_
joblib.dump(model,'models/rf_model.joblib')
print('Saved ../models/rf_model.joblib')